In [1]:
# loading env variables

from dotenv import load_dotenv
load_dotenv(override = True)

import os
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")


python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 11
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 13
python-dotenv could not parse statement starting at line 15


In [3]:
# document loading

from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("./ai_papers")

pdf_documents = loader.load()

print(len(pdf_documents))
print(pdf_documents[0])

11
page_content='Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to t

In [4]:
# chunking

from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 200)

pdf_chunks = splitter.split_documents(pdf_documents)

print(len(pdf_chunks))
print(len(pdf_chunks[1].page_content))
print(f"pdf_chunk 2 : ", pdf_chunks[2])

106
492
pdf_chunk 2 :  page_content='mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including' metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an 

In [50]:
# Embedding

from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

# this embedding generates # of dimensions = 384
# embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2") 

'''
Embedding model              Dimension
────────────────────────────────────────
all-MiniLM-L6-v2                  384
all-mpnet-base-v2                 768
'''
embedding = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

# this embedding generates # of dimensions = 1536 
# embedding = OpenAIEmbeddings(model="text-embedding-3-small")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2832.87it/s]


In [51]:
print(embedding.model_name)

sentence-transformers/all-MiniLM-L6-v2


In [ ]:
import os
api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY is not set")

Index 'pinecone-vectordb' is ready.


In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=api_key)

index_name = "pinecone-vectordb"  # pinecone_vectordb is invalid

# Delete old index
pc.delete_index(index_name)

# Recreate for hybrid search
pc.create_index(
    name=index_name,
    dimension=384,       # must match your embedding dimension
    metric="dotproduct",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "pinecone-vectordb",
    "metric": "dotproduct",
    "host": "pinecone-vectordb-lbphvci.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [56]:
db = PineconeVectorStore.from_documents(pdf_chunks,
                        embedding = embedding,
                        index_name = index_name)

In [57]:
db.similarity_search("What is transformer?")

[Document(id='3267fa48-4976-4b59-ad96-eaf925fec1e5', metadata={'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'creationdate': '', 'creator': 'PyPDF', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requi

In [58]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder().default()

In [59]:
index = pc.Index(index_name)

In [76]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(
    embeddings=embedding,
    sparse_encoder=bm25_encoder,
    index=index,
    text_key="text"
)

In [77]:
from langchain_groq import ChatGroq

llm = ChatGroq(model = "openai/gpt-oss-120b")

In [79]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question only from the context.

Context:
{context}

Question:
{input}
""")

In [80]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)

In [81]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [82]:
response = retrieval_chain.invoke({"input": "what is transformer?"})

In [83]:
response

{'input': 'what is transformer?',
 'context': [Document(metadata={'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'creationdate': '', 'creator': 'PyPDF', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and re

In [85]:
retrieval_chain.invoke({"input": "Who wrote this research paper?"})

{'input': 'Who wrote this research paper?',
 'context': [Document(metadata={'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'creationdate': '', 'creator': 'PyPDF', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleliza